# Assignment 3: Sales Dashboard Portal
**Course:** Fundamentals of Programming for Business (CL1002)

**Group members:**
- Noman Sajid Toor (25L-5210)
- Mahad Ahmad (25L-5277)

A console portal to add, view and search sales and see a sales report. Sales are saved in `sales.csv`.

## 1. Imports and settings

In [1]:
import csv
import numpy as np
import pandas as pd

FILE = "sales.csv"
FIELDS = ["id", "date", "product", "region", "quantity", "price"]
USERS = {"admin": "1234"}

## 2. Login

In [2]:
def user_login(username, password):
    """Return True if the username and password are correct."""
    return USERS.get(username) == password

## 3. File handling
Loads the CSV at startup and saves it after every change. Errors are handled with try/except/finally.

In [3]:
def load_sales(filename):
    """Read sales from a CSV file and return a list of dictionaries."""
    sales = []
    try:
        with open(filename, "r") as f:
            for row in csv.DictReader(f):
                row["id"] = int(row["id"])
                row["quantity"] = int(row["quantity"])
                row["price"] = float(row["price"])
                sales.append(row)
    except FileNotFoundError:
        print("File not found. Starting with no records.")
    except PermissionError:
        print("No permission to read the file.")
    except ValueError:
        print("The file has bad data.")
    finally:
        print("Records loaded:", len(sales))
    return sales


def save_sales(sales, filename):
    """Write all sales to the CSV file."""
    try:
        with open(filename, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=FIELDS)
            writer.writeheader()
            writer.writerows(sales)
    except PermissionError:
        print("Cannot save. Close the file if it is open.")

## 4. Add, search and view

In [4]:
def add_record(sales, product, region, quantity, price, date):
    """Add a sale to the list, save it, and return it."""
    sale = {"id": len(sales) + 1, "date": date, "product": product.title(),
            "region": region.title(), "quantity": quantity, "price": price}
    sales.append(sale)
    save_sales(sales, FILE)
    return sale


def search_record(sales, word):
    """Return the sales that match the word in product, region or date."""
    found = []
    for s in sales:
        if word.lower() in (s["product"] + s["region"] + s["date"]).lower():
            found.append(s)
    return found


def show_sales(sales):
    """Print the sales."""
    for s in sales:
        print(s["id"], s["date"], s["product"], s["region"], s["quantity"], s["price"])

## 5. Report (Pandas and NumPy)

In [5]:
def generate_report(sales):
    """Print a summary of the sales and return the DataFrame."""
    if len(sales) == 0:
        print("No sales yet.")
        return None
    df = pd.DataFrame(sales)
    df["total"] = df["quantity"] * df["price"]
    print("Total revenue:", df["total"].sum())
    print("Average sale:", round(np.mean(df["total"]), 2))
    print("Best product:", df.groupby("product")["total"].sum().idxmax())
    print("\nRevenue by region:")
    print(df.groupby("region")["total"].sum())
    return df

## 6. Main menu
The `while True` loop keeps the portal running until the user exits. `get_number()` uses try/except so wrong input does not crash it.

In [6]:
def get_number(message):
    """Keep asking until the user types a number."""
    while True:
        try:
            return float(input(message))
        except ValueError:
            print("Please enter a number.")


def main():
    logged_in = False
    for i in range(3):
        if user_login(input("Username: "), input("Password: ")):
            logged_in = True
            break
        print("Wrong login.")
    if not logged_in:
        print("Access denied.")
        return

    sales = load_sales(FILE)
    while True:
        print("\n1. Add sale  2. View sales  3. Search  4. Report  5. Exit")
        choice = input("Choose: ")
        if choice == "1":
            product = input("Product: ")
            region = input("Region: ")
            quantity = int(get_number("Quantity: "))
            price = get_number("Price: ")
            date = input("Date (YYYY-MM-DD): ")
            add_record(sales, product, region, quantity, price, date)
            print("Sale added.")
        elif choice == "2":
            show_sales(sales)
        elif choice == "3":
            show_sales(search_record(sales, input("Search: ")))
        elif choice == "4":
            generate_report(sales)
        elif choice == "5":
            print("Goodbye!")
            break
        else:
            print("Choose 1 to 5.")

## 7. Sample test runs

In [7]:
# Test 1: login
print(user_login("admin", "1234"))
print(user_login("admin", "wrong"))

True
False


In [8]:
# Test 2: load file and add sample sales (creates sales.csv)
sales = load_sales(FILE)
if len(sales) == 0:
    add_record(sales, "Laptop", "Lahore", 3, 150000, "2026-11-01")
    add_record(sales, "Mouse", "Karachi", 20, 1500, "2026-11-02")
    add_record(sales, "Monitor", "Lahore", 5, 32000, "2026-11-05")
    add_record(sales, "Laptop", "Islamabad", 2, 150000, "2026-11-08")
show_sales(sales)

File not found. Starting with no records.
Records loaded: 0
1 2026-11-01 Laptop Lahore 3 150000
2 2026-11-02 Mouse Karachi 20 1500
3 2026-11-05 Monitor Lahore 5 32000
4 2026-11-08 Laptop Islamabad 2 150000


In [9]:
# Test 3: search
show_sales(search_record(sales, "lahore"))

1 2026-11-01 Laptop Lahore 3 150000
3 2026-11-05 Monitor Lahore 5 32000


In [10]:
# Test 4: report
df = generate_report(sales)

Total revenue: 940000
Average sale: 235000.0
Best product: Laptop

Revenue by region:
region
Islamabad    300000
Karachi       30000
Lahore       610000
Name: total, dtype: int64


In [11]:
# Test 5: missing file and wrong input
print(load_sales("missing.csv"))
try:
    quantity = int("ten")
except ValueError:
    print("ValueError caught: quantity must be a number.")

File not found. Starting with no records.
Records loaded: 0
[]
ValueError caught: quantity must be a number.


## 8. Run the portal
Login: `admin` / `1234`. Remove the `#` and run the cell.

In [12]:
# main()